# ClaimsIQ 02 — LangGraph Claims Workflow, Connected to Snowflake via MCP

**This notebook:** Rebuilds the Day 11 Planner/Executor/Validator
LangGraph — but every node now calls the REAL Snowflake data through
`mcp_snowflake_server.py`, instead of the `MOCK_ORDERS` dictionary. The
graph structure itself doesn't change at all.

### Prerequisite
Run `00_snowflake_setup_and_seed_data.ipynb` and
`01_mcp_server_snowflake.ipynb` first, in this Jupyter environment
(so `mcp_snowflake_server.py` exists on disk).

## Step 1 — Install & import

In [ ]:
%pip install -q langgraph openai snowflake-connector-python

In [ ]:
import os, json
from typing import TypedDict, List, Optional
from openai import OpenAI
from langgraph.graph import StateGraph, END
from mcp_snowflake_server import claims_server, SimpleMCPClient

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before continuing"
client = OpenAI()

mcp_client = SimpleMCPClient(claims_server)
mcp_client.connect()
print("LangGraph agent connected to the Snowflake MCP server.")

## Step 2 — Define the shared state

Three new fields beyond Day 11's `ClaimState`: `fraud_signals`,
`velocity`, and `peer_comparison` — the extra data a REAL fraud
decision needs, that the mock version never had.

In [ ]:
class ClaimState(TypedDict):
    customer_id: str
    claim_id: str
    profile: dict
    claim_details: dict
    fraud_signals: list
    velocity: dict
    peer_comparison: dict
    decision: str
    reasoning: str

print("ClaimState defined.")

In [ ]:
CURRENT_VALIDATOR_PROMPT = "prompts/validator_v2.txt"
 
def load_prompt(path):
    with open(path) as f:
        return f.read().strip()
 
validator_prompt = load_prompt(CURRENT_VALIDATOR_PROMPT)

## Step 3 — Write the nodes

Five nodes now, not three — Planner, an Executor for EACH data source
the investigation needs, and a Validator that synthesizes everything.
This is deliberately closer to a real system: one Executor step per
distinct Snowflake query the case genuinely requires.

In [ ]:
def planner_node(state: ClaimState) -> dict:
    prompt = (
        f"A warranty claim (claim_id={state['claim_id']}, customer_id={state['customer_id']}) "
        "needs a full investigation: customer profile, claim details, fraud signals, "
        "transaction velocity, and peer-normalized spend comparison. "
        "List these as 4-5 short subtasks."
    )
    resp = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}], temperature=0)
    print(f"[planner] {resp.choices[0].message.content[:200]}...")
    return {}

def executor_profile_node(state: ClaimState) -> dict:
    result = mcp_client.call_tool("get_customer_profile", customer_id=state["customer_id"])
    print(f"[executor:profile] {result}")
    return {"profile": result}

def executor_claim_node(state: ClaimState) -> dict:
    result = mcp_client.call_tool("get_claim_details", claim_id=state["claim_id"])
    print(f"[executor:claim] {result}")
    return {"claim_details": result}

def executor_fraud_node(state: ClaimState) -> dict:
    signals = mcp_client.call_tool("check_fraud_signals", customer_id=state["customer_id"])
    velocity = mcp_client.call_tool("get_transaction_velocity", customer_id=state["customer_id"])
    peer = mcp_client.call_tool("compare_to_peer_spend", customer_id=state["customer_id"])
    print(f"[executor:fraud] signals={signals}, velocity={velocity}, peer={peer}")
    return {"fraud_signals": signals, "velocity": velocity, "peer_comparison": peer}

print("Nodes defined.")

In [ ]:
def validator_node(state: ClaimState) -> dict:
    prompt = f"""You are reviewing a warranty claim. Synthesize ALL the evidence below
and decide: APPROVED, DENIED, or ESCALATED (approve the claim but flag the
account for human fraud review).

CUSTOMER PROFILE: {state['profile']}
CLAIM DETAILS: {state['claim_details']}
FRAUD SIGNALS (last 30 days): {state['fraud_signals']}
TRANSACTION VELOCITY (last 48h): {state['velocity']}
PEER SPEND COMPARISON (7d): {state['peer_comparison']}

Remember: high fraud signals do NOT automatically mean the claim itself is
fraudulent — a claim can be entirely legitimate while the account still
deserves a closer look. Weigh customer tenure, claim plausibility, AND
fraud signals together, and explicitly consider whether the peer
comparison suggests the spend spike is personal or market-wide (e.g. a
sale event).

Respond in this exact format:
DECISION: <APPROVED|DENIED|ESCALATED>
REASONING: <2-3 sentences citing the specific evidence above>"""

    resp = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}], temperature=0)
    text = resp.choices[0].message.content
    print(f"[validator] {text}")

    decision = "ESCALATED"
    for line in text.split("\n"):
        if line.startswith("DECISION:"):
            decision = line.replace("DECISION:", "").strip()
    return {"decision": decision, "reasoning": text}

print("validator_node defined.")

## Step 4 — Build the graph

Planner → three parallel-in-spirit Executors (run sequentially here for
simplicity) → Validator → END. No conditional branching needed this
time — the Validator's OWN output carries the three-way decision,
rather than the graph routing differently per case.

In [ ]:
graph = StateGraph(ClaimState)
graph.add_node("planner", planner_node)
graph.add_node("executor_profile", executor_profile_node)
graph.add_node("executor_claim", executor_claim_node)
graph.add_node("executor_fraud", executor_fraud_node)
graph.add_node("validator", validator_node)

graph.set_entry_point("planner")
graph.add_edge("planner", "executor_profile")
graph.add_edge("executor_profile", "executor_claim")
graph.add_edge("executor_claim", "executor_fraud")
graph.add_edge("executor_fraud", "validator")
graph.add_edge("validator", END)

app = graph.compile()
print("Graph compiled — 5 nodes, all data pulled live from Snowflake.")

## Step 5 — Run Ananya Rao's case

In [ ]:
result = app.invoke({"customer_id": "CUST99001", "claim_id": "CLM99001"})

print("\n" + "=" * 60)
print("FINAL DECISION:", result["decision"])
print("=" * 60)
print(result["reasoning"])

## Deliverable

1. The full node-by-node trace and final decision from Step 5.
2. Did the Validator correctly distinguish “this account has real fraud
   signals” from “this specific claim is fraudulent”? Quote the exact
   reasoning that shows (or fails to show) that distinction.
3. One paragraph: this graph replaced Day 11's single Executor with
   THREE separate executor nodes, one per data source. What would break
   if you tried to squeeze all three Snowflake calls into a single
   executor node instead? (Hint: think about what Day 13's tracing
   would look like either way.)